In [ ]:
! pip install transformers datasets torch
! pip install --upgrade tensorflow_hub
! pip install evaluate

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from datasets import Dataset, load_dataset
import torch
import tensorflow_hub as hub

In [ ]:
# import dataset
dataset = pd.read_csv('../dataset/cleaned_dataset_bert.csv')
dataset.shape

(51093, 2)

In [4]:
dataset.head()

,processed_text,status
0,oh my gosh,Anxiety
1,trouble sleeping confused mind restless heart ...,Anxiety
2,all wrong back off dear forward doubt stay in ...,Anxiety
3,i have shifted my focus to something else but ...,Anxiety
4,i am restless and restless it is been a month ...,Anxiety


In [5]:
dataset['status'].value_counts()

status
Normal                  16040
Depression              15094
Suicidal                10644
Anxiety                  3623
Bipolar                  2501
Stress                   2296
Personality disorder      895
Name: count, dtype: int64

In [6]:
# label encoding
unique_labels = sorted(dataset['status'].unique())
unique_labels

['Anxiety',
 'Bipolar',
 'Depression',
 'Normal',
 'Personality disorder',
 'Stress',
 'Suicidal']

In [7]:
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

dataset['label'] = dataset['status'].map(label_to_id)
print(f"Unique Classes: {unique_labels}")
print(f"Label Mapping: {label_to_id}")
print("-" * 40)

dataset.head()

Unique Classes: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
Label Mapping: {'Anxiety': 0, 'Bipolar': 1, 'Depression': 2, 'Normal': 3, 'Personality disorder': 4, 'Stress': 5, 'Suicidal': 6}
----------------------------------------


,processed_text,status,label
0,oh my gosh,Anxiety,0
1,trouble sleeping confused mind restless heart ...,Anxiety,0
2,all wrong back off dear forward doubt stay in ...,Anxiety,0
3,i have shifted my focus to something else but ...,Anxiety,0
4,i am restless and restless it is been a month ...,Anxiety,0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    dataset['processed_text'],
    dataset['label'],
    stratify=dataset['label']
)

In [9]:
%pip install torch==2.1.0 torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118
%pip install transformers==4.36.2
%pip install datasets==2.15.0
%pip install accelerate==0.25.0

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0, 2.2.0+cu118, 2.2.1, 2.2.1+cu118, 2.2.2, 2.2.2+cu118, 2.3.0, 2.3.0+cu118, 2.3.1, 2.3.1+cu118, 2.4.0, 2.4.0+cu118, 2.4.1, 2.4.1+cu118, 2.5.0, 2.5.0+cu118, 2.5.1, 2.5.1+cu118, 2.6.0, 2.6.0+cu118, 2.7.0, 2.7.0+cu118, 2.7.1, 2.7.1+cu118, 2.8.0, 2.9.0, 2.9.1)
ERROR: No matching distribution found for torch==2.1.0


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.



In [10]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W1112 16:58:35.100000 18540 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\Yohan\anaconda3\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
c:\Users\Yohan\anaconda3\Lib\site-packages\huggingface_hub\file_download

In [11]:
dataset.head()

,processed_text,status,label
0,oh my gosh,Anxiety,0
1,trouble sleeping confused mind restless heart ...,Anxiety,0
2,all wrong back off dear forward doubt stay in ...,Anxiety,0
3,i have shifted my focus to something else but ...,Anxiety,0
4,i am restless and restless it is been a month ...,Anxiety,0


In [12]:
X = dataset['processed_text'].values
y = dataset['label'].values

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
text_column_name = 'processed_text'
label_column_name = "status"

#model_name = "distilbert-base-uncased"
model_name = "bert-base-uncased"
text_size = 0.2
num_labels = 7

In [15]:
df_train, df_test = train_test_split(
    dataset,
    test_size=text_size
)

In [16]:
from datasets import Dataset

In [17]:
train_dataset = Dataset.from_pandas(df_train)
test_dataset = Dataset.from_pandas(df_test)

In [20]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\Yohan\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [23]:
def preprocess_function(examples):
    return tokenizer(
        examples['processed_text'],
        truncation=True,
        padding="max_length")


In [24]:
tokenizer_train = train_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/40874 [00:00<?, ? examples/s]

TypeError: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]